In [ ]:
using GeoMakie, GLMakie, NCDatasets, Dates, Missings, GeometryBasics, Downloads, Shapefile, ZipFile, Interpolations

: 

In [ ]:
data_dir = "extracted_chl"
chl_files = sort(filter(f -> startswith(basename(f), "chl") && endswith(f, ".nc"), readdir(data_dir, join=true)))
coord_files = sort(filter(f -> startswith(basename(f), "geo") && endswith(f, ".nc"), readdir(data_dir, join=true)))

# Load datasets into memory (read arrays and close files to avoid HDF/NetCDF driver issues)
datasets_chl = []
for f in chl_files
    try
        ds = NCDataset(f)
        data = ds["CHL_NN"][:]
        creation_time = haskey(ds.attrib, "creation_time") ? ds.attrib["creation_time"] : "unknown"
        push!(datasets_chl, (data = data, attrib = Dict("creation_time" => creation_time)))
        close(ds)
    catch e
        println("Error loading $f: $e")
    end
end

datasets_coord = []
for f in coord_files
    try
        ds = NCDataset(f)
        lat = ds["latitude"][:]
        lon = ds["longitude"][:]
        alt = ds["altitude"][:]
        push!(datasets_coord, (lat = lat, lon = lon, alt = alt))
        close(ds)
    catch e
        println("Error loading $f: $e")
    end
end

println("Loaded $(length(datasets_chl)) CHL files and $(length(datasets_coord)) coordinate files")

In [ ]:
# Define a struct for the dataset
struct CHLDataset
    chl::Array{Float32, 2}
    lat::Array{Float32, 2}
    lon::Array{Float32, 2}
    alt::Array{Float32, 2}
    creation_time::String
end

day1_data::Array{CHLDataset, 1} = CHLDataset[]

In [ ]:
# 1. Configuration
const COLS, ROWS = 4865, 4091 
step = 5 
erie_lon_lims = (-83.6, -78.2)
erie_lat_lims = (41.3, 43.1)

# --- Helper: Safe Coordinate Extraction ---
function to_gpu_matrix_final(data_vec, is_lon=false)
    # Reshape to Matrix
    mat = reshape(data_vec, COLS, ROWS)' 
    sub = mat[1:step:end, 1:step:end]
    res = Float32.(coalesce.(sub, NaN))
    
    # Normalize Longitude
    if is_lon
        res = [(!isnan(x) && x > 180) ? x - 360 : x for x in res]
    end
    return res
end

# --- Helper: Gentle Data Cleaning (No Strict Altitude) ---
function clean_chl_v3(chl_vec)
    # Reshape Data
    chl_mat = reshape(chl_vec, COLS, ROWS)'[1:step:end, 1:step:end]
    data_out = Float32.(coalesce.(chl_mat, NaN))
    
    # Filter: Basic Sanity Check Only
    # We removed the altitude check to ensure data appears.
    # We only remove obvious sensor errors (extreme spikes or negatives).
    for i in eachindex(data_out)
        val = data_out[i]
        if !isnan(val)
            # Log scale: -5.0 (0.00001) to 3.0 (1000.0)
            if val < -5.0f0 || val > 3.0f0 
                 data_out[i] = NaN32
            end
        end
    end

    return 10.0f0 .^ data_out
end

# --- Helper: Lakes Geometry ---
function get_lakes_geo()
    url = "https://naciscdn.org/naturalearth/10m/physical/ne_10m_lakes.zip"
    zip_name = "ne_10m_lakes.zip"
    if !isfile(zip_name)
        Downloads.download(url, zip_name)
    end
    r = ZipFile.Reader(zip_name)
    for f in r.files
        if endswith(f.name, ".shp") || endswith(f.name, ".shx") || endswith(f.name, ".dbf") || endswith(f.name, ".prj")
             write(f.name, read(f))
        end
    end
    return Shapefile.shapes(Shapefile.Table("ne_10m_lakes.shp"))
end
lakes_geo = get_lakes_geo()

# 2. Setup Data (INITIAL FRAME)
# Wrappers for Dynamic Updates
lats = Observable(to_gpu_matrix_final(datasets_coord[1].lat, false))
lons = Observable(to_gpu_matrix_final(datasets_coord[1].lon, true))
chl_data = Observable(clean_chl_v3(datasets_chl[1].data)) # Note: No alt arg needed now
plot_title = Observable("Lake Erie CHL: $(datasets_chl[1].attrib["creation_time"])")

# 3. Figure Setup
fig = Figure(size = (1200, 850))
ax = GeoAxis(fig[1, 1], dest = "+proj=merc", title = plot_title)

# Plot Land (Base Layer)
poly!(ax, GeoMakie.land(), color = :grey50)

# Plot Lakes (Water Layer) - Dark Background for the data
poly!(ax, lakes_geo, color = :grey30) 

# Plot Data Surface (Middle Layer)
plt = Makie.surface!(ax, lons, lats, chl_data; 
               shading = NoShading, colormap = :algae, 
               colorrange = (0.1, 50.0), colorscale = log10)
translate!(plt, 0, 0, 10)

# Plot Lake Borders (Top Layer - Outlines)
lns = poly!(ax, lakes_geo, color = :transparent, strokecolor = :white, strokewidth = 2.0)
translate!(lns, 0, 0, 20)

# Window Limits
limits!(ax, erie_lon_lims, erie_lat_lims)

# Add Town Labels
for (name, lon, lat) in [("Cleveland", -81.69, 41.50), ("Toledo", -83.55, 41.65), 
                         ("Erie", -80.08, 42.13), ("Buffalo", -78.88, 42.89)]
    m = scatter!(ax, Point2f(lon, lat), color = :red, markersize = 12, strokecolor = :white, strokewidth = 1)
    t = text!(ax, Point2f(lon + 0.05, lat + 0.03), text = name, color = :white, fontsize = 14, strokecolor = :black, strokewidth = 2)
    translate!(m, 0, 0, 20); translate!(t, 0, 0, 20)
end
Colorbar(fig[1, 2], plt, label = "Chlorophyll-a (mg/m³)")

# 4. Record Loop (The Fix)
record(fig, "lake_erie_chlorophyll.mp4", 1:length(datasets_chl); framerate = 2) do i
    plot_title[] = "Lake Erie CHL: $(datasets_chl[i].attrib["creation_time"])"
    
    # UPDATE COORDINATES: This fixes the "Ghost/Sliding" data
    lons[] = to_gpu_matrix_final(datasets_coord[i].lon, true)
    lats[] = to_gpu_matrix_final(datasets_coord[i].lat, false)
    
    # UPDATE DATA
    chl_data[] = clean_chl_v3(datasets_chl[i].data)
    
    println("Frame $i rendered.")
end

Frame 1 rendered.
Frame 2 rendered.
Frame 3 rendered.
Frame 4 rendered.
Frame 5 rendered.
Frame 6 rendered.
Frame 7 rendered.


"lake_erie_chlorophyll.mp4"

In [64]:
# Configuration for interpolation
minutes_per_interval = 30  # Target time interval in minutes

# Function to parse creation_time toDateTime
function parse_creation_time(time_str)
    if time_str == "unknown"
        return nothing
    end
    try
        return DateTime(time_str, "yyyy-mm-ddTHH:MM:SS.sssZ")
    catch
        try
            return DateTime(time_str, "yyyy-mm-ddTHH:MM:SSZ")
        catch
            return nothing
        end
    end
end

# Calculate interpolated frames
function create_interpolated_frames(datasets_chl, datasets_coord, target_minutes)
    interp_frames = []
    
    for i in 1:(length(datasets_chl)-1)
        # Add original frame
        push!(interp_frames, (
            chl_idx = i,
            coord_idx = i,
            alpha = 0.0,
            time_str = datasets_chl[i].attrib["creation_time"]
        ))
        
        # Parse timestamps
        t1 = parse_creation_time(datasets_chl[i].attrib["creation_time"])
        t2 = parse_creation_time(datasets_chl[i+1].attrib["creation_time"])
        
        if !isnothing(t1) && !isnothing(t2)
            time_diff_minutes = Dates.value(t2 - t1) / (1000 * 60)  # Convert ms to minutes
            n_interp = Int(floor(time_diff_minutes / target_minutes)) - 1
            
            # Create interpolated frames
            for j in 1:n_interp
                alpha = j / (n_interp + 1)
                interp_time = t1 + Dates.Minute(Int(round(j * time_diff_minutes / (n_interp + 1))))
                push!(interp_frames, (
                    chl_idx = i,
                    coord_idx = i,
                    alpha = alpha,
                    time_str = Dates.format(interp_time, "yyyy-mm-ddTHH:MM:SS.000Z")
                ))
            end
        end
    end
    
    # Add last frame
    push!(interp_frames, (
        chl_idx = length(datasets_chl),
        coord_idx = length(datasets_coord),
        alpha = 0.0,
        time_str = datasets_chl[end].attrib["creation_time"]
    ))
    
    return interp_frames
end

interp_frames = create_interpolated_frames(datasets_chl, datasets_coord, minutes_per_interval)
println("Generated $(length(interp_frames)) total frames (including interpolated)")

# Update visualization with interpolation
record(fig, "lake_erie_chlorophyll_smooth.mp4", interp_frames; framerate = 4) do frame
    plot_title[] = "Lake Erie CHL: $(frame.time_str)"
    
    i = frame.chl_idx
    alpha = frame.alpha
    
    if alpha == 0.0
        # Original frame
        lons[] = to_gpu_matrix_final(datasets_coord[i].lon, true)
        lats[] = to_gpu_matrix_final(datasets_coord[i].lat, false)
        chl_data[] = clean_chl_v3(datasets_chl[i].data)
    else
        # Interpolated frame
        i_next = min(i + 1, length(datasets_chl))
        
        # Interpolate coordinates
        lon1 = to_gpu_matrix_final(datasets_coord[i].lon, true)
        lon2 = to_gpu_matrix_final(datasets_coord[i_next].lon, true)
        lons[] = (1 - alpha) .* lon1 .+ alpha .* lon2
        
        lat1 = to_gpu_matrix_final(datasets_coord[i].lat, false)
        lat2 = to_gpu_matrix_final(datasets_coord[i_next].lat, false)
        lats[] = (1 - alpha) .* lat1 .+ alpha .* lat2
        
        # Interpolate CHL data
        chl1 = clean_chl_v3(datasets_chl[i].data)
        chl2 = clean_chl_v3(datasets_chl[i_next].data)
        chl_data[] = (1 - alpha) .* chl1 .+ alpha .* chl2
    end
    
    println("Frame $(frame.time_str) rendered (alpha=$(alpha))")
end

Generated 427 total frames (including interpolated)
Frame 2023-08-17T04:58:51Z rendered (alpha=0.0)
Frame 2023-08-17T05:29:51.000Z rendered (alpha=0.022222222222222223)
Frame 2023-08-17T05:59:51.000Z rendered (alpha=0.044444444444444446)
Frame 2023-08-17T06:30:51.000Z rendered (alpha=0.06666666666666667)
Frame 2023-08-17T07:00:51.000Z rendered (alpha=0.08888888888888889)
Frame 2023-08-17T07:31:51.000Z rendered (alpha=0.1111111111111111)
Frame 2023-08-17T08:02:51.000Z rendered (alpha=0.13333333333333333)
Frame 2023-08-17T08:32:51.000Z rendered (alpha=0.15555555555555556)
Frame 2023-08-17T09:03:51.000Z rendered (alpha=0.17777777777777778)
Frame 2023-08-17T09:34:51.000Z rendered (alpha=0.2)
Frame 2023-08-17T10:04:51.000Z rendered (alpha=0.2222222222222222)
Frame 2023-08-17T10:35:51.000Z rendered (alpha=0.24444444444444444)
Frame 2023-08-17T11:05:51.000Z rendered (alpha=0.26666666666666666)
Frame 2023-08-17T11:36:51.000Z rendered (alpha=0.28888888888888886)
Frame 2023-08-17T12:07:51.000Z r

"lake_erie_chlorophyll_smooth.mp4"